## Project Overview

This notebook trains binary classifiers for Type 2 Diabetes detection from tongue images with 5-fold cross-validation.

- **Input**: `pool/<class>/` and `manifest.csv` from `tongue-dataprep` in `cv` mode (`cfg.manifest`)
- **Grouping schemes**: `photo` (each photo is a group, leaks other photos of the same person), `block` (10 consecutive file numbers), `cluster` (identity clusters from image similarity)
- **Per fold k**: test = fold k originals, validation = fold (k+1) mod 5 originals (early stopping, checkpoint, threshold), training = the other three folds with their offline augmentations
- **Runs** (`CONFIGS`): ResNet-50 under all three schemes; MobileNetV3-Large and EfficientNet-B2 under `cluster`; ResNet-50 on the periphery only (tongue disc masked out) under `cluster`
- **Outputs**: `oof_probs.csv` (one row per test image per run), `cv_summary.csv`, `reliability.png`, Grad-CAM images
- **Kaggle sessions**: set `RUN_ONLY` (comma-separated `CONFIGS` indices) to split the runs; runs already in `oof_probs.csv` are skipped
- **Local smoke test**: `SMOKE=1` trains fold 0 of every config for one epoch per stage on three batches, without pretrained weights

## 1. Configuration & Paths

In [ ]:
!pip install grad-cam

In [ ]:
# 0. Suppress Environment Noise
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

# 1. Primary engine imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import transforms
from torchvision import models as tv_models
from torchvision.models import ResNet50_Weights, MobileNet_V3_Large_Weights

import timm
import sklearn
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

# 2. Support & Utility imports
import re
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)

@dataclass
class CFG:
    manifest: str = os.environ.get('MANIFEST', '/kaggle/input/tonguedatapreped/datapreped/manifest.csv')
    output_dir: str = os.environ.get('OUTPUT_BASE', '/kaggle/working')
    smoke: bool = os.environ.get('SMOKE') == '1'

    use_circular_mask: bool = True
    use_background_jitter: bool = True
    use_cutout: bool = True
    padding_range: float = 0.25

    class_to_label: Dict[str, int] = None

    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 2
    pin_memory: bool = True

    warmup_epochs: int = 10
    finetune_epochs: int = 50
    head_lr: float = 1e-3
    ft_lr: float = 1e-4
    eta_min: float = 1e-6
    weight_decay: float = 1e-4
    early_stopping_patience: int = 12

    mixup_alpha: float = 0.2
    seed: int = 42
    n_folds: int = 5

cfg = CFG()
cfg.class_to_label = {'diabetes': 1, 'nondiabetes': 0}
if cfg.smoke:  # local CPU check of the whole loop, not a result
    cfg.warmup_epochs, cfg.finetune_epochs, cfg.num_workers = 1, 1, 0

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Manifest:', cfg.manifest)
print('Versions: torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| timm', timm.__version__, '| sklearn', sklearn.__version__)

## 2. Dataset & DataLoaders

In [ ]:
class CircularMaskTransform:
    """Soft circular mask on the normalized tensor; invert=True keeps only the periphery (shortcut probe)."""
    def __init__(self, mode='test', invert=False):
        self.mode = mode
        self.invert = invert
    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        c, h, w = img.shape
        y, x = torch.meshgrid(torch.linspace(-1, 1, h), torch.linspace(-1, 1, w), indexing='ij')
        dist = torch.sqrt(x**2 + y**2)
        mask = 1.0 - torch.clamp((dist - 0.85) / 0.1, 0, 1)
        if self.invert:  # corners outside the inscribed circle: lips, clothing, background, little tongue
            mask = torch.clamp((dist - 1.0) / 0.1, 0, 1)
        if self.mode == 'train' and cfg.use_background_jitter:
            noise = (torch.rand_like(img) * 2.0 - 1.0)
            return img * mask + noise * (1.0 - mask)
        return img * mask

class TongueDataset(Dataset):
    """Rows of manifest.csv; returns (image, label, manifest file path)."""
    def __init__(self, rows: pd.DataFrame, root: Path, transform=None, train=False):
        self.samples = list(zip(rows['file'], rows['label'].astype(int)))
        self.root, self.transform, self.train = root, transform, train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file, label = self.samples[idx]
        image = Image.open(self.root / file).convert('RGB')
        if self.train and random.random() < 0.6:
            w, h = image.size
            pad = int(min(w, h) * random.uniform(0.05, cfg.padding_range))
            image = transforms.Pad(pad, fill=0)(image)
        if self.transform is not None:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32), file

def make_tfms(input_mode: str):
    """(train, eval) transforms; input_mode 'tongue' keeps the central disc, 'periphery' only the outside."""
    invert = input_mode == 'periphery'
    identity = transforms.Lambda(lambda x: x)
    norm = [transforms.Resize((cfg.img_size, cfg.img_size)), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
    train = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.03),
        *norm,
        CircularMaskTransform('train', invert) if cfg.use_circular_mask else identity,
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.12), ratio=(0.3, 3.3), value='random') if cfg.use_cutout else identity,
    ])
    evaluation = transforms.Compose([*norm, CircularMaskTransform('test', invert) if cfg.use_circular_mask else identity])
    return train, evaluation

def make_loaders(train_rows, val_rows, test_rows, input_mode):
    train_tfms, eval_tfms = make_tfms(input_mode)
    sample_weights = 1.0 / train_rows['label'].map(train_rows['label'].value_counts()).values
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(train_rows), replacement=True)
    kw = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    return (DataLoader(TongueDataset(train_rows, image_root, train_tfms, train=True), sampler=sampler, **kw),
            DataLoader(TongueDataset(val_rows, image_root, eval_tfms), shuffle=False, **kw),
            DataLoader(TongueDataset(test_rows, image_root, eval_tfms), shuffle=False, **kw))

manifest = pd.read_csv(cfg.manifest)
image_root = Path(cfg.manifest).parent
print(f'Manifest: {len(manifest)} images, {int(manifest.is_orig.sum())} originals')

## 3. Model Definitions

### Backbone choices

The dataset has about 250 unique images per class, so all three models are ImageNet-pretrained CNNs fine-tuned only in their last stages.

*   **EfficientNet-B2** (7.7 M parameters with the new head): compound-scaled, sample-efficient on small image sets. Fine-tuning unfreezes its last two blocks plus `conv_head` and `bn2`, the final 1x1 convolution that sits between the blocks and the classifier.
*   **ResNet-50** (23.5 M): the common baseline in tongue-image studies. Fine-tuning unfreezes `layer3` and `layer4`.
*   **MobileNetV3-Large** (4.2 M): small and fast, the candidate for mobile deployment. Fine-tuning unfreezes the last three feature blocks, which include its final convolution.

Vision transformers are not used: they typically need far larger training sets than a few hundred unique images.

In [ ]:
def build_model(name: str) -> nn.Module:
    if name == 'efficientnet_b2':
        model = timm.create_model('efficientnet_b2', pretrained=not cfg.smoke)
        model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(1408, 1))
        return model
    if name == 'resnet50':
        model = tv_models.resnet50(weights=None if cfg.smoke else ResNet50_Weights.DEFAULT)
        model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2048, 1))
        return model
    if name == 'mobilenet_v3_large':
        model = tv_models.mobilenet_v3_large(weights=None if cfg.smoke else MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[3] = nn.Sequential(nn.Dropout(0.4), nn.Linear(1280, 1))
        return model
    raise ValueError(f'Unsupported model name: {name}')

def freeze_all_backbone(model: nn.Module, name: str):
    for p in model.parameters():
        p.requires_grad = False
    if name == 'efficientnet_b2':
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif name == 'resnet50':
        for p in model.fc.parameters():
            p.requires_grad = True
    elif name == 'mobilenet_v3_large':
        for p in model.classifier.parameters():
            p.requires_grad = True

def partial_unfreeze(model: nn.Module, name: str):
    freeze_all_backbone(model, name)
    if name == 'efficientnet_b2':
        # conv_head + bn2 sit between the blocks and the classifier; left frozen they stay at ImageNet weights
        for m in (*model.blocks[-2:], model.conv_head, model.bn2):
            for p in m.parameters():
                p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif name == 'resnet50':
        # Unfreeze layer3 AND layer4 to let ResNet adapt better generalizing features
        for p in model.layer3.parameters():
            p.requires_grad = True
        for p in model.layer4.parameters():
            p.requires_grad = True
        for p in model.fc.parameters():
            p.requires_grad = True
    elif name == 'mobilenet_v3_large':
        for p in model.features[-3:].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True

def get_trainable_params(model: nn.Module):
    return [p for p in model.parameters() if p.requires_grad]

def get_gradcam_target_layer(model: nn.Module, name: str):
    if name == 'efficientnet_b2': return [model.blocks[-1]]
    if name == 'resnet50': return [model.layer4[-1]]
    if name == 'mobilenet_v3_large': return [model.features[-1]]
    raise ValueError(f'Unsupported model name: {name}')

model_names = ['efficientnet_b2', 'resnet50', 'mobilenet_v3_large']

## 4. Training Loop
Incorporating **Focal Loss** for strict penalty on missed complex minority textures.

### Training strategy

1.  **Focal loss** ($\gamma=2$, $\alpha=0.6$ for T2D): down-weights easy examples so training focuses on hard ones, and weights missed T2D cases more.
2.  **Two-stage transfer learning**: a warm-up trains only the new head on a frozen backbone; fine-tuning then unfreezes the late backbone stages at a lower learning rate.
3.  **Mixup** during fine-tuning: blends pairs of images and their labels as a regularizer.
4.  **Selection on validation only**: the checkpoint with the highest validation PR-AUC is kept, ties broken by lower validation loss (PR-AUC often saturates at 1.0 on small validation folds).

In [ ]:
class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.6, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce_loss).mean()

criterion = FocalLossWithLogits(alpha=0.6, gamma=2.0).to(device)
use_amp = device.type == 'cuda'

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def evaluate(model, loader, device):
    model.eval()
    losses, probs, labels, paths = [], [], [], []
    with torch.no_grad():
        for x, y, p in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(x).squeeze(1)
                loss = criterion(logits, y)
            pred_prob = torch.sigmoid(logits.float()).detach().cpu().numpy()
            losses.append(loss.item())
            probs.extend(pred_prob.tolist())
            labels.extend(y.detach().cpu().numpy().tolist())
            paths.extend(p)

    mean_loss = float(np.mean(losses)) if losses else 0.0
    labels_arr, probs_arr = np.array(labels), np.array(probs)
    auc = roc_auc_score(labels_arr, probs_arr) if len(np.unique(labels_arr)) > 1 else 0.5
    ap = average_precision_score(labels_arr, probs_arr) if len(np.unique(labels_arr)) > 1 else 0.5
    return {'loss': mean_loss, 'auc': float(auc), 'ap': float(ap), 'labels': labels_arr, 'probs': probs_arr, 'paths': paths}

def train_one_model(model_name: str, train_loader, val_loader, best_path: Path):
    model = build_model(model_name).to(device)
    freeze_all_backbone(model, model_name)
    optimizer = optim.AdamW(get_trainable_params(model), lr=cfg.head_lr, weight_decay=cfg.weight_decay)
    scheduler, scaler = None, torch.amp.GradScaler('cuda', enabled=use_amp)
    total_epochs = cfg.warmup_epochs + cfg.finetune_epochs
    best_key, epochs_no_improve = (-1.0, -np.inf), 0

    for epoch in range(1, total_epochs + 1):
        if epoch == cfg.warmup_epochs + 1:
            partial_unfreeze(model, model_name)
            optimizer = optim.AdamW(get_trainable_params(model), lr=cfg.ft_lr, weight_decay=cfg.weight_decay)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.finetune_epochs, eta_min=cfg.eta_min)
        use_mixup = epoch > cfg.warmup_epochs
        model.train()
        for step, (x, y, _) in enumerate(train_loader):
            if cfg.smoke and step == 3:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                if use_mixup:
                    x_mixed, y_a, y_b, lam = mixup_data(x, y, alpha=cfg.mixup_alpha)
                    logits = model(x_mixed).squeeze(1)
                    loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
                else:
                    logits = model(x).squeeze(1)
                    loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if scheduler is not None: scheduler.step()
        val_out = evaluate(model, val_loader, device)
        print(f'  epoch {epoch:02d}/{total_epochs} | val_PR_AUC={val_out["ap"]:.4f} val_loss={val_out["loss"]:.4f}')
        key = (val_out['ap'], -val_out['loss'])  # PR-AUC first; saturated PR-AUC ties go to lower loss
        if key > best_key:
            best_key, epochs_no_improve = key, 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= cfg.early_stopping_patience: break

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    return model

## 5. Cross-Validated Training

Each run trains one model per fold and appends its test-fold predictions to `oof_probs.csv`.

In [ ]:
def binary_metrics_from_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'ROC-AUC': float(roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5),
        'PR-AUC (AP)': float(average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5),
        'F1': float(f1_score(y_true, y_pred, average='binary', zero_division=0)),
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'Precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'Recall': float(recall_score(y_true, y_pred, zero_division=0)),
    }

def choose_threshold(labels, probs):
    """Maximize recall subject to precision >= 0.90 on validation; ties go to the middle of the tied thresholds,
    which keeps a margin on both sides when validation is separated perfectly."""
    thresholds = np.round(np.arange(0.10, 0.901, 0.01), 2)
    scores = np.array([(precision_score(labels, probs >= t, zero_division=0) >= 0.90) * recall_score(labels, probs >= t, zero_division=0)
                       for t in thresholds])
    if scores.max() == 0:
        return 0.50  # fallback when no threshold reaches 90% precision on validation
    tied = np.flatnonzero(scores == scores.max())
    return float(thresholds[tied[len(tied) // 2]])

assert choose_threshold(np.array([0, 0, 1, 1]), np.array([0.1, 0.2, 0.6, 0.7])) == 0.41  # plateau 0.21-0.60 -> its middle

def run_fold(scheme: str, fold: int, model_name: str, input_mode: str) -> pd.DataFrame:
    k, val_fold = f'fold_{scheme}', (fold + 1) % cfg.n_folds
    test_rows = manifest[(manifest[k] == fold) & (manifest['is_orig'] == 1)]
    val_rows = manifest[(manifest[k] == val_fold) & (manifest['is_orig'] == 1)]
    train_rows = manifest[~manifest[k].isin([fold, val_fold])]
    assert (val_rows['is_orig'] == 1).all() and (test_rows['is_orig'] == 1).all()
    fams = [set(r['family']) for r in (train_rows, val_rows, test_rows)]
    assert fams[0].isdisjoint(fams[1]) and fams[0].isdisjoint(fams[2]) and fams[1].isdisjoint(fams[2]), 'family in two parts'
    assert val_rows['label'].nunique() == 2 and test_rows['label'].nunique() == 2

    seed_everything(cfg.seed)
    train_loader, val_loader, test_loader = make_loaders(train_rows, val_rows, test_rows, input_mode)
    ckpt = Path(cfg.output_dir) / f'{scheme}_{model_name}_{input_mode}_fold{fold}.pt'
    model = train_one_model(model_name, train_loader, val_loader, ckpt)
    val_out = evaluate(model, val_loader, device)
    threshold = choose_threshold(val_out['labels'], val_out['probs'])
    test_out = evaluate(model, test_loader, device)
    print(f'{scheme}/{model_name}/{input_mode} fold {fold}: val AUC {val_out["auc"]:.3f}, '
          f'test AUC {test_out["auc"]:.3f}, threshold {threshold:.2f}')
    if (scheme, model_name, input_mode, fold) != GRADCAM_RUN:
        ckpt.unlink()  # keep only the checkpoint the Grad-CAM section uses
    out = pd.DataFrame({'scheme': scheme, 'model': model_name, 'input': input_mode, 'fold': fold,
                        'file': test_out['paths'], 'label': test_out['labels'].astype(int),
                        'prob': test_out['probs'].round(5), 'threshold': threshold})
    return out.merge(manifest[['file', 'group_cluster']], on='file', validate='one_to_one')

CONFIGS = [('photo', 'resnet50', 'tongue'), ('block', 'resnet50', 'tongue'), ('cluster', 'resnet50', 'tongue'),
           ('cluster', 'mobilenet_v3_large', 'tongue'), ('cluster', 'efficientnet_b2', 'tongue'),
           ('cluster', 'resnet50', 'periphery')]
GRADCAM_RUN = ('cluster', 'resnet50', 'tongue', 0)
# Kaggle: list the CONFIGS indices to train in this session ([] = all). Attach the output of earlier sessions as
# input datasets: every oof_probs.csv under PRIOR_OOF_DIR is merged in and its runs are skipped.
RUN_ONLY = []
run_only = [int(i) for i in os.environ.get('RUN_ONLY', '').split(',') if i.strip()] or RUN_ONLY or range(len(CONFIGS))
folds = [0] if cfg.smoke else range(cfg.n_folds)

oof_path = Path(cfg.output_dir) / 'oof_probs.csv'
prior_dir = Path(os.environ.get('PRIOR_OOF_DIR', '/kaggle/input'))
prior = sorted(prior_dir.rglob('oof_probs.csv')) if prior_dir.exists() else []
frames = [pd.read_csv(p) for p in prior + ([oof_path] if oof_path.exists() else [])]
done = set()
if frames:
    merged = pd.concat(frames, ignore_index=True).drop_duplicates(['scheme', 'model', 'input', 'fold', 'file'])
    merged.to_csv(oof_path, index=False)
    done = set(map(tuple, merged[['scheme', 'model', 'input', 'fold']].drop_duplicates().values.tolist()))
    print(f'Merged {len(prior)} earlier oof_probs.csv file(s): {len(done)} (config, fold) runs already done')
for i in run_only:
    scheme, model_name, input_mode = CONFIGS[i]
    for fold in folds:
        if (scheme, model_name, input_mode, fold) in done:
            print('skip (already in oof_probs.csv):', scheme, model_name, input_mode, fold)
            continue
        rows = run_fold(scheme, fold, model_name, input_mode)
        assert len(rows) == int(((manifest[f'fold_{scheme}'] == fold) & (manifest['is_orig'] == 1)).sum())
        rows.to_csv(oof_path, mode='a', header=not oof_path.exists(), index=False)

oof = pd.read_csv(oof_path)
print(oof.groupby(['scheme', 'model', 'input'])['fold'].nunique().rename('folds done'))

## 6. Cross-Validated Results

In [ ]:
# Pooled out-of-fold results, identity-cluster bootstrap intervals, leakage ladder, calibration, colour baseline
def _cluster_draws(df, B, seed):
    """Index arrays of B stratified bootstrap samples that resample identity clusters within each class."""
    rng = np.random.default_rng(seed)
    groups = {c: [g.index.values for _, g in df[df.label == c].groupby('group_cluster')] for c in (0, 1)}
    for _ in range(B):
        yield np.concatenate([groups[c][j] for c in (0, 1) for j in rng.integers(0, len(groups[c]), len(groups[c]))])

def cluster_bootstrap_auc(df, B=2000, seed=42):
    """Pooled ROC-AUC and 95% interval from resampling identity clusters (one person is one unit)."""
    df = df.reset_index(drop=True)
    aucs = [roc_auc_score(df.label.values[i], df.prob.values[i]) for i in _cluster_draws(df, B, seed)]
    return (float(roc_auc_score(df.label, df.prob)), *np.percentile(aucs, [2.5, 97.5]).tolist())

def paired_delta_auc(df_a, df_b, B=2000, seed=42):
    """AUC(a) - AUC(b) on the same images, 95% interval from resampling identity clusters."""
    both = df_a.merge(df_b[['file', 'prob']], on='file', suffixes=('', '_b'), validate='one_to_one').reset_index(drop=True)
    d = lambda i: roc_auc_score(both.label.values[i], both.prob.values[i]) - roc_auc_score(both.label.values[i], both.prob_b.values[i])
    deltas = [d(i) for i in _cluster_draws(both, B, seed)]
    return (float(d(np.arange(len(both)))), *np.percentile(deltas, [2.5, 97.5]).tolist())

import cv2
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Self-checks on synthetic data: perfect scores give AUC 1 with a degenerate interval; identical runs give delta 0
_syn = pd.DataFrame({'file': [f'f{i}' for i in range(40)], 'label': [0, 1] * 20,
                     'group_cluster': [f'c{i // 4}_{i % 2}' for i in range(40)]})
_syn['prob'] = _syn['label'].astype(float)
assert cluster_bootstrap_auc(_syn, B=200) == (1.0, 1.0, 1.0)
_syn['prob'] = np.random.default_rng(0).random(40)
assert paired_delta_auc(_syn, _syn, B=200) == (0.0, 0.0, 0.0)

def color_baseline(manifest, scheme='cluster'):
    """HSV histogram of the central disc + logistic regression, same folds and threshold rule as the CNNs."""
    orig = manifest[manifest.is_orig == 1].reset_index(drop=True)
    yy, xx = np.mgrid[-1:1:cfg.img_size * 1j, -1:1:cfg.img_size * 1j]
    disc = ((xx ** 2 + yy ** 2) <= 0.85 ** 2).astype(np.uint8)
    def feat(file):
        hsv = cv2.cvtColor(np.array(Image.open(image_root / file).convert('RGB').resize((cfg.img_size, cfg.img_size))), cv2.COLOR_RGB2HSV)
        h = cv2.calcHist([hsv], [0, 1, 2], disc, [16, 8, 8], [0, 180, 0, 256, 0, 256]).ravel()
        return h / h.sum()
    X, y, k = np.stack([feat(f) for f in orig.file]), orig.label.values, orig[f'fold_{scheme}'].values
    rows = []
    for fold in range(cfg.n_folds):
        val_fold = (fold + 1) % cfg.n_folds
        tr, va, te = ~np.isin(k, [fold, val_fold]), k == val_fold, k == fold
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)).fit(X[tr], y[tr])
        threshold = choose_threshold(y[va], clf.predict_proba(X[va])[:, 1])
        rows.append(pd.DataFrame({'scheme': scheme, 'model': 'color_hist', 'input': 'tongue', 'fold': fold,
                                  'file': orig.file[te].values, 'label': y[te], 'prob': clf.predict_proba(X[te])[:, 1],
                                  'threshold': threshold, 'group_cluster': orig.group_cluster[te].values}))
    return pd.concat(rows, ignore_index=True)

results = pd.concat([oof, color_baseline(manifest)], ignore_index=True)
originals = set(manifest.loc[manifest.is_orig == 1, 'file'])
summary = []
for (scheme, model_name, input_mode), run in results.groupby(['scheme', 'model', 'input']):
    assert run.file.is_unique, f'{scheme}/{model_name}/{input_mode}: an image tested twice'
    complete = run.fold.nunique() == cfg.n_folds
    if complete:
        assert set(run.file) == originals, f'{scheme}/{model_name}/{input_mode}: OOF does not cover every original once'
    fold_auc = run.groupby('fold').apply(lambda g: roc_auc_score(g.label, g.prob))
    auc, lo, hi = cluster_bootstrap_auc(run)
    person = run.groupby('group_cluster').agg(label=('label', 'first'), prob=('prob', 'mean'))
    pred = run.prob >= run.threshold
    sens, spec = float(pred[run.label == 1].mean()), float((~pred)[run.label == 0].mean())
    ppv = lambda prev: sens * prev / den if (den := sens * prev + (1 - spec) * (1 - prev)) > 0 else float('nan')
    summary.append({'scheme': scheme, 'model': model_name, 'input': input_mode, 'folds': run.fold.nunique(),
                    'auc_fold_mean': fold_auc.mean(), 'auc_fold_sd': fold_auc.std(ddof=1), 'auc_pooled': auc,
                    'auc_lo': lo, 'auc_hi': hi,
                    'auc_person': roc_auc_score(person.label, person.prob) if person.label.nunique() == 2 else np.nan,
                    'sens': sens, 'spec': spec, 'ppv_11.1': ppv(0.111), 'ppv_5': ppv(0.05),
                    'brier': float(np.mean((run.prob - run.label) ** 2))})
summary = pd.DataFrame(summary).round(3)
summary.to_csv(Path(cfg.output_dir) / 'cv_summary.csv', index=False)
print(summary.to_string(index=False))

# Leakage ladder: paired AUC differences between grouping schemes (ResNet-50, tongue input)
deltas = []
res = lambda s: results[(results.scheme == s) & (results.model == 'resnet50') & (results.input == 'tongue')]
complete = lambda s: res(s).fold.nunique() == cfg.n_folds or (cfg.smoke and len(res(s)) > 0)
for a, b in (('photo', 'cluster'), ('block', 'cluster'), ('photo', 'block')):
    if not (complete(a) and complete(b)):
        print(f'delta {a} vs {b} skipped: a scheme has not finished all {cfg.n_folds} folds')
    else:
        d, lo, hi = paired_delta_auc(res(a), res(b))
        deltas.append({'a': a, 'b': b, 'n_images': len(res(a).merge(res(b), on='file')), 'delta_auc': d, 'lo': lo, 'hi': hi})
deltas = pd.DataFrame(deltas).round(3)
deltas.to_csv(Path(cfg.output_dir) / 'cv_deltas.csv', index=False)
print(deltas.to_string(index=False))

# Reliability of the pooled out-of-fold probabilities
plt.figure(figsize=(6, 6)); plt.plot([0, 1], [0, 1], 'k:', label='perfect')
for (scheme, model_name, input_mode), run in results[results.scheme == 'cluster'].groupby(['scheme', 'model', 'input']):
    frac, mean = calibration_curve(run.label, run.prob, n_bins=10, strategy='quantile')
    plt.plot(mean, frac, 'o-', label=f'{model_name} ({input_mode})')
plt.xlabel('Mean predicted probability'); plt.ylabel('Observed T2D fraction'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Reliability, cluster folds, pooled out-of-fold'); plt.savefig(Path(cfg.output_dir) / 'reliability.png', dpi=200); plt.show()

## 7. Grad-CAM

In [ ]:
# Grad-CAM for the cluster-scheme ResNet-50 of fold 0: first three test images per class and the
# highest-scoring false positive from each run of consecutive file numbers.
def visualize_debug_cam(model, files, title, model_name='resnet50'):
    _, eval_tfms = make_tfms('tongue')
    cam = GradCAM(model=model, target_layers=get_gradcam_target_layer(model, model_name))
    fig, axes = plt.subplots(len(files), 3, figsize=(15, 5 * len(files)), squeeze=False)
    for i, file in enumerate(files):
        pil = Image.open(image_root / file).convert('RGB').resize((cfg.img_size, cfg.img_size))
        inp = eval_tfms(pil).unsqueeze(0).to(device)
        masked_np = inp.squeeze(0).cpu().permute(1, 2, 0).numpy()
        masked_np = np.clip(masked_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
        grayscale_cam = cam(input_tensor=inp, targets=[BinaryClassifierOutputTarget(1)])[0]
        overlay = show_cam_on_image(masked_np.astype(np.float32), grayscale_cam, use_rgb=True)
        with torch.no_grad(): prob = torch.sigmoid(model(inp)).item()
        axes[i, 0].imshow(np.array(pil)); axes[i, 0].set_title(Path(file).name[-30:]); axes[i, 0].axis('off')
        axes[i, 1].imshow(masked_np); axes[i, 1].set_title('Masked model input'); axes[i, 1].axis('off')
        axes[i, 2].imshow(overlay); axes[i, 2].set_title(f'Heatmap (prob {prob:.3f})'); axes[i, 2].axis('off')
    plt.tight_layout(); plt.savefig(Path(cfg.output_dir) / f'gradcam_{title}.png', dpi=200); plt.show()

ckpt = Path(cfg.output_dir) / '{}_{}_{}_fold{}.pt'.format(*GRADCAM_RUN)
if ckpt.exists():
    scheme, model_name, input_mode, fold = GRADCAM_RUN
    cam_model = build_model(model_name).to(device)
    cam_model.load_state_dict(torch.load(ckpt, map_location=device)); cam_model.eval()
    run = oof[(oof.scheme == scheme) & (oof.model == model_name) & (oof.input == input_mode) & (oof.fold == fold)]
    visualize_debug_cam(cam_model, run[run.label == 1].file.head(3).tolist() + run[run.label == 0].file.head(3).tolist(), 'cv_fold0_test')
    file_num = lambda f: int(re.search(r'__orig__r_n?d__(\d+)_', Path(f).name).group(1))
    fp_files = []
    for f in run[(run.label == 0) & (run.prob >= run.threshold)].sort_values('prob', ascending=False).file:
        if all(abs(file_num(f) - file_num(g)) > 2 for g in fp_files):
            fp_files.append(f)
    print('False positives (fold 0):', len(run[(run.label == 0) & (run.prob >= run.threshold)]))
    if fp_files:
        visualize_debug_cam(cam_model, fp_files[:3], 'cv_fold0_false_positives')
else:
    print('Grad-CAM skipped: run', GRADCAM_RUN, 'not trained in this session')